In [1]:
import pandas as pd 
hourly = pd.read_parquet("../data/processed/hourly_jan2026.parquet")["trips"]

In [2]:
import sys
sys.path.append("..")
from app.backtest import walk_forward_splits   


In [3]:
for train_idx, test_idx in walk_forward_splits(hourly, 24*14, 24, 24, "expanding"):
    print(train_idx.min(), train_idx.max(), " | ", test_idx.min(), test_idx.max())
    assert train_idx.max() < test_idx.min()

2026-01-01 00:00:00 2026-01-14 23:00:00  |  2026-01-15 00:00:00 2026-01-15 23:00:00
2026-01-01 00:00:00 2026-01-15 23:00:00  |  2026-01-16 00:00:00 2026-01-16 23:00:00
2026-01-01 00:00:00 2026-01-16 23:00:00  |  2026-01-17 00:00:00 2026-01-17 23:00:00
2026-01-01 00:00:00 2026-01-17 23:00:00  |  2026-01-18 00:00:00 2026-01-18 23:00:00
2026-01-01 00:00:00 2026-01-18 23:00:00  |  2026-01-19 00:00:00 2026-01-19 23:00:00
2026-01-01 00:00:00 2026-01-19 23:00:00  |  2026-01-20 00:00:00 2026-01-20 23:00:00
2026-01-01 00:00:00 2026-01-20 23:00:00  |  2026-01-21 00:00:00 2026-01-21 23:00:00
2026-01-01 00:00:00 2026-01-21 23:00:00  |  2026-01-22 00:00:00 2026-01-22 23:00:00
2026-01-01 00:00:00 2026-01-22 23:00:00  |  2026-01-23 00:00:00 2026-01-23 23:00:00
2026-01-01 00:00:00 2026-01-23 23:00:00  |  2026-01-24 00:00:00 2026-01-24 23:00:00
2026-01-01 00:00:00 2026-01-24 23:00:00  |  2026-01-25 00:00:00 2026-01-25 23:00:00
2026-01-01 00:00:00 2026-01-25 23:00:00  |  2026-01-26 00:00:00 2026-01-26 2

In [4]:
from sklearn.model_selection import TimeSeriesSplit

tscv = TimeSeriesSplit(n_splits=17, test_size=24)
for train_i, test_i in tscv.split(hourly):
    print(f"  Train: index={train_i}")
    print(f"  Test:  index={test_i}")

  Train: index=[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242

In [5]:
print(len(hourly))
for train_idx, test_idx in walk_forward_splits(hourly, 24*14, 24, 24):
    print(test_idx.min(), test_idx.max())

744
2026-01-15 00:00:00 2026-01-15 23:00:00
2026-01-16 00:00:00 2026-01-16 23:00:00
2026-01-17 00:00:00 2026-01-17 23:00:00
2026-01-18 00:00:00 2026-01-18 23:00:00
2026-01-19 00:00:00 2026-01-19 23:00:00
2026-01-20 00:00:00 2026-01-20 23:00:00
2026-01-21 00:00:00 2026-01-21 23:00:00
2026-01-22 00:00:00 2026-01-22 23:00:00
2026-01-23 00:00:00 2026-01-23 23:00:00
2026-01-24 00:00:00 2026-01-24 23:00:00
2026-01-25 00:00:00 2026-01-25 23:00:00
2026-01-26 00:00:00 2026-01-26 23:00:00
2026-01-27 00:00:00 2026-01-27 23:00:00
2026-01-28 00:00:00 2026-01-28 23:00:00
2026-01-29 00:00:00 2026-01-29 23:00:00
2026-01-30 00:00:00 2026-01-30 23:00:00
2026-01-31 00:00:00 2026-01-31 23:00:00


In [6]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

results = []

for train_idx, test_idx in walk_forward_splits(hourly, 24*14, 24, 24):
    y_true = hourly.loc[test_idx]
    y_pred = hourly.loc[test_idx - pd.Timedelta(hours=168)]

    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)

    results.append({"mae": mae, "mape": mape})

results

[{'mae': 903.25, 'mape': 0.13891920709796857},
 {'mae': 468.0833333333333, 'mape': 0.09987922486824223},
 {'mae': 774.0416666666666, 'mape': 0.11884972567705192},
 {'mae': 432.6666666666667, 'mape': 0.09218368602791506},
 {'mae': 997.2083333333334, 'mape': 0.36401152270354364},
 {'mae': 344.5416666666667, 'mape': 0.07640717805372514},
 {'mae': 321.8333333333333, 'mape': 0.07609165730060709},
 {'mae': 321.5833333333333, 'mape': 0.0820882253082885},
 {'mae': 516.2083333333334, 'mape': 0.07553153696375754},
 {'mae': 713.5, 'mape': 0.11948226312475389},
 {'mae': 3129.25, 'mape': 2.1376883657443155},
 {'mae': 1311.5416666666667, 'mape': 1.1223033813974261},
 {'mae': 651.7916666666666, 'mape': 0.13876950965288562},
 {'mae': 594.375, 'mape': 0.11702910498270157},
 {'mae': 456.7916666666667, 'mape': 0.08688429854598952},
 {'mae': 485.1666666666667, 'mape': 0.09257308039696821},
 {'mae': 508.125, 'mape': 0.08017812633305749}]

In [7]:
pd.DataFrame(results).mean()

pd.DataFrame(results).agg(["mean", "median", "std"])

,mae,mape
mean,760.585784,0.295228
median,516.208333,0.099879
std,664.987675,0.537693
